## Part 2 | Data quality log

> Before trusting any number, check the data. List every issue you find in a table: issue, how you detected it, how many rows/users it affects, how you handled it, and its impact on the results. Include the logic you used to map raw `utm_source` values to channels.

| Issue | How we found it | Affected | Fix and impact |
|---|---|---|---|
| Apparent QA accounts | Matched trimmed, case-folded names against six reviewed test-name patterns and checked linked transactions | 30 users and 682 transactions after deduplication | Excluded by user ID. This removed **EGP 38.59M** from deduplicated successful Q3 UTC deposits. QA status was inferred from the available evidence. |
| Exact transaction duplicates | Compared all transaction fields | 577 of 42,378 raw rows | Kept one copy of each row. This removed **EGP 991,800** from successful Q3 UTC deposits. |
| Inconsistent UTM formatting | Compared raw UTM values with trimmed, lowercased values | 1,274 clean users had a source value change | Preserved the original values and added normalized fields so equivalent sources map consistently. |
| Missing acquisition source | Counted blank or null sources and checked nonblank values against the alias list | 794 raw users and 764 after QA exclusion. Every nonblank normalized source matched an alias. | Kept these users in overall reporting and classified their source as **Unknown** in channel reporting. They receive no paid-channel attribution. |
| Reporting time zone | Reconciled successful deposits on the UTC and Cairo reporting calendars using the case’s fixed UTC+2 offset | After deduplication and QA exclusion, the Q3 total changed from 7,001 deposits in UTC to 6,995 in Cairo | Use Cairo for calendar reporting and elapsed time for activation windows. The Cairo adjustment reduced Q3 deposit volume by **EGP 2,300**. |
| Platform-reported installs and attributed signups measure different steps | Compared platform install totals with signups attributed through normalized `utm_source` | Meta reported **3,355 installs** and had **2,094 attributed signups**. Google Ads reported **1,803 installs** and had **1,427 attributed signups**. | Use reported installs for CPI and attributed signups for signup-to-activation rates. The totals come from different sources, so their gap does not establish install overcounting. |
| Deposit volume concentrated on September 3 | Grouped cleaned successful deposits by Cairo reporting day and inspected the transaction rows | **EGP 10.297M** across **89 deposits** on September 3, or **51.76%** of cleaned Q3 volume | Keep qualifying deposits in the official **EGP 19.89M** total. Show **EGP 9.60M** excluding September 3 as a sensitivity and ask Finance to identify what drove the spike. |
| Some September loans had yet to reach DPD30 | Compared each loan’s due date plus 30 days with the November 5 extract date | **270 of 623** September loans were immature | Keep all loans in disbursal reporting. Use only mature loans in the DPD30 denominator. September was **34/353 (9.63%)** at DPD30, versus **73/623 (11.72%)** short of full repayment at extract. |

> QA account discovery. While investigating duplicate transactions, I linked the affected user IDs to the users table and noticed names that looked like test accounts. I then used an LLM to review an exported list of distinct user names and flag likely QA names. The notebook applies six reviewed name patterns to identify matching user IDs and exclude those accounts and their transactions. QA status was inferred from the names.

### Acquisition channel mapping

Trim whitespace and lowercase the raw `utm_source`, then apply this mapping.

| Channel | Normalized aliases |
|---|---|
| Meta | `meta`, `facebook ads`, `facebook`, `fb`, `instagram`, `ig` |
| Google Ads | `google`, `google ads`, `adwords`, `gads` |
| TikTok | `tiktok`, `tt` |
| Referral | `referral`, `ref`, `invite` |
| Organic | `organic` |
| Direct | `(direct)` |
| Unknown | Blank, null, or any value outside the alias list |

The paid-channel comparison groups Referral, Organic, and Direct as **Other**.

### Validation and findings

The cleaned exports contain **6,200 users** and **41,119 transactions**. Loans, Plus subscriptions, CVM assignments, and marketing spend retained all **1,310**, **111**, **697**, and **276** rows, respectively.

The validation cell found zero missing key values, repeated keys, or unmatched user links across the tables checked. All **137** holdout assignments had blank send timestamps, as expected. The **82** blank Plus end dates indicate subscriptions open at extract. Blank KYC verification and last repayment dates were preserved.

**What carries into the analysis.** QA exclusions explain most of the deposit correction. The KPI notebook shows how sensitive the headline is to September 3, the marketing notebook keeps install and activation measures separate, and the credit notebook compares loans at the same repayment checkpoint. Missing UTMs remain an unresolved attribution issue.

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    folder for folder in (Path.cwd(), Path.cwd().parent)
    if (folder / "data" / "raw" / "Data & Insights - Analyst Case.xlsx").is_file()
)
WORKBOOK = ROOT / "data" / "raw" / "Data & Insights - Analyst Case.xlsx"
CLEANED = ROOT / "data" / "cleaned"
Q3_START = "2026-07-01"
Q3_END = "2026-10-01"
QA_NAME_PATTERNS = {
    "qa lucky", "test user", "test qa", "test test",
    "تجربه تطبيق", "تجربة",
}
CLEANED.mkdir(exist_ok=True)

In [4]:
# Read the six data sheets from the original workbook.
sheets = pd.read_excel(
    WORKBOOK,
    sheet_name=["users", "transactions", "loans", "plus_subscriptions",
                "cvm_campaigns", "marketing_spend"],
    engine="openpyxl",
)
users_raw = sheets["users"]
transactions_raw = sheets["transactions"]
loans_raw = sheets["loans"]
plus_subscriptions_raw = sheets["plus_subscriptions"]
cvm_campaigns_raw = sheets["cvm_campaigns"]
marketing_spend_raw = sheets["marketing_spend"]

In [5]:
# Users: find exact copies and derive QA IDs from the reviewed name patterns.
duplicate_users = users_raw.duplicated()
unique_users = users_raw.loc[~duplicate_users]
normalized_names = unique_users["full_name"].str.strip().str.casefold()
qa_mask = normalized_names.isin(QA_NAME_PATTERNS)
qa_users = unique_users.loc[qa_mask, ["user_id", "full_name"]]
qa_user_ids = qa_users["user_id"].unique()
users = unique_users.loc[~qa_mask].copy()

# Standardize UTM strings while keeping the original values for traceability.
users["utm_source_normalized"] = users["utm_source"].str.strip().str.lower().replace("", pd.NA)
users["utm_campaign_normalized"] = users["utm_campaign"].str.strip().str.lower().replace("", pd.NA)

print(f"Exact duplicate users removed: {duplicate_users.sum():,} / {len(users_raw):,}")
print(f"QA users removed: {len(qa_user_ids):,}")
display(qa_users.sort_values("user_id"))
print(f"Clean users: {len(users):,}")
users.to_csv(CLEANED / "users.csv", index=False)


# Review input for a semantic pass over every user.
transaction_counts = transactions_raw.groupby("user_id")["txn_id"].nunique()
qa_review = unique_users[["user_id", "full_name"]].copy()
qa_review["distinct_transaction_count"] = qa_review["user_id"].map(transaction_counts).fillna(0).astype(int)
qa_review.to_csv(CLEANED / "qa_user_review.csv", index=False)

Exact duplicate users removed: 0 / 6,230
QA users removed: 30


,user_id,full_name
6082,106201,QA Lucky
3796,106202,test test
4714,106203,QA Lucky
4121,106204,Test User
1681,106205,تجربه تطبيق
3076,106206,QA Lucky
796,106207,test qa
1726,106208,Test User
3399,106209,QA Lucky
4249,106210,Test User


Clean users: 6,200


In [6]:
# Transactions: remove exact copies first so QA rows are counted once.
duplicate_transactions = transactions_raw.duplicated()
unique_transactions = transactions_raw.loc[~duplicate_transactions]
qa_transaction_mask = unique_transactions["user_id"].isin(qa_user_ids)
transactions = unique_transactions.loc[~qa_transaction_mask].copy()

print(f"Exact copies removed: {duplicate_transactions.sum():,} / {len(transactions_raw):,}")
print(f"QA rows removed: {qa_transaction_mask.sum():,} / {len(unique_transactions):,} deduplicated rows")
print(f"Clean transactions: {len(transactions):,}")
transactions.to_csv(CLEANED / "transactions.csv", index=False)


# Save the first four steps of the Task 3.1 deposit bridge.
q3_deposits = transactions_raw.loc[
    transactions_raw["txn_type"].eq("deposit")
    & transactions_raw["txn_ts"].ge(Q3_START)
    & transactions_raw["txn_ts"].lt(Q3_END)
]
successful_deposits = q3_deposits.loc[q3_deposits["status"].eq("success")]
unique_deposits = successful_deposits.drop_duplicates()
valid_deposits = unique_deposits.loc[unique_deposits["user_id"].isin(users["user_id"])]
audit_steps = [
    ("Raw UTC, all statuses", q3_deposits),
    ("Raw UTC, successful", successful_deposits),
    ("Remove exact copies", unique_deposits),
    ("Keep valid non-QA users", valid_deposits),
]
deposit_audit = pd.DataFrame(
    [(step, label, len(rows), rows["amount_egp"].sum())
     for step, (label, rows) in enumerate(audit_steps, start=1)],
    columns=["step", "scope", "deposits", "volume_egp"],
)
deposit_audit.to_csv(CLEANED / "q3_deposit_audit.csv", index=False)

Exact copies removed: 577 / 42,378
QA rows removed: 682 / 41,801 deduplicated rows
Clean transactions: 41,119


In [7]:
# Loans: profiling found no rows needing removal.
loans = loans_raw.copy()
print("Blank last repayment dates are preserved.")
print(f"Clean loans: {len(loans):,} / {len(loans_raw):,} rows")
loans.to_csv(CLEANED / "loans.csv", index=False)

Blank last repayment dates are preserved.
Clean loans: 1,310 / 1,310 rows


In [8]:
# Plus subscriptions: profiling found no rows needing removal.
plus_subscriptions = plus_subscriptions_raw.copy()
print("Blank end dates indicate ongoing subscriptions.")
print(f"Clean plus subscriptions: {len(plus_subscriptions):,} / {len(plus_subscriptions_raw):,} rows")
plus_subscriptions.to_csv(CLEANED / "plus_subscriptions.csv", index=False)

Blank end dates indicate ongoing subscriptions.
Clean plus subscriptions: 111 / 111 rows


In [9]:
# Cvm campaigns: profiling found no rows needing removal.
cvm_campaigns = cvm_campaigns_raw.copy()
print("Blank send timestamps are expected for holdout users.")
print(f"Clean cvm campaigns: {len(cvm_campaigns):,} / {len(cvm_campaigns_raw):,} rows")
cvm_campaigns.to_csv(CLEANED / "cvm_campaigns.csv", index=False)

Blank send timestamps are expected for holdout users.
Clean cvm campaigns: 697 / 697 rows


In [10]:
# Marketing spend: profiling found no rows needing removal.
marketing_spend = marketing_spend_raw.copy()
print("Platform-reported installs are preserved as supplied.")
print(f"Clean marketing spend: {len(marketing_spend):,} / {len(marketing_spend_raw):,} rows")
marketing_spend.to_csv(CLEANED / "marketing_spend.csv", index=False)

Platform-reported installs are preserved as supplied.
Clean marketing spend: 276 / 276 rows


In [11]:
# Validate the cleaned tables and report expected blanks.
tables = {
    "users": (users, ["user_id"]),
    "transactions": (transactions, ["txn_id"]),
    "loans": (loans, ["loan_id"]),
    "plus_subscriptions": (plus_subscriptions, ["user_id", "plus_start_date"]),
    "cvm_campaigns": (cvm_campaigns, ["user_id"]),
    "marketing_spend": (marketing_spend, ["channel", "date"]),
}

validation = []
for name, (table, key) in tables.items():
    validation.append({
        "table": name,
        "key": ", ".join(key),
        "rows with missing key values": table[key].isna().any(axis=1).sum(),
        "rows beyond the first per key": table.duplicated(subset=key).sum(),
        "rows with unmatched user IDs": (
            (~table["user_id"].isin(users["user_id"])).sum()
            if name != "users" and "user_id" in table.columns
            else pd.NA
        ),
    })

display(pd.DataFrame(validation))

missing_send = cvm_campaigns["sent_ts"].isna()
holdout = cvm_campaigns["group"].eq("holdout")

display(pd.DataFrame([
    {
        "check": "Holdout rows with no send timestamp (expected)",
        "rows": (holdout & missing_send).sum(),
    },
    {
        "check": "Non-holdout rows with no send timestamp (review)",
        "rows": (~holdout & missing_send).sum(),
    },
    {
        "check": "Holdout rows with a send timestamp (review)",
        "rows": (holdout & ~missing_send).sum(),
    },
    {
        "check": "Plus rows with no end date (ongoing at extract)",
        "rows": plus_subscriptions["plus_end_date"].isna().sum(),
    },
]))

,table,key,rows with missing key values,rows beyond the first per key,rows with unmatched user IDs
0,users,user_id,0,0,<NA>
1,transactions,txn_id,0,0,0
2,loans,loan_id,0,0,0
3,plus_subscriptions,"user_id, plus_start_date",0,0,0
4,cvm_campaigns,user_id,0,0,0
5,marketing_spend,"channel, date",0,0,<NA>


,check,rows
0,Holdout rows with no send timestamp (expected),137
1,Non-holdout rows with no send timestamp (review),0
2,Holdout rows with a send timestamp (review),0
3,Plus rows with no end date (ongoing at extract),82
